# Data Processing

This notebook basically parses the summarised data to be used in the training. It extracts the useful information and writes it to a csv file that looks like this:

| datalen_bytes | pub_count | sub_count | best_effort | multicast | durability | mean_latency_us | mean_total_throughput_mbps | mean_sample_rate | samples_received | samples_lost |
|---------------|-----------|-----------|-------------|-----------|------------|-----------------|----------------------------|------------------|------------------|--------------|
| ...           | ...       | ...       | ...         | ...       | ...        | ...             | ...                        | ...              | ...              | ...          |

The plan is to use the following as input:
- `datalen_bytes`
- `pub_count`
- `sub_count`
- `best_effort`
- `multicast`
- `durability`

And the following as outputs:
- `mean_latency_us`
- `mean_total_throughput_mbps`
- `mean_sample_rate`
- `samples_received`
- `samples_lost`


In [1]:
import os
import pandas as pd
import re
from rich.progress import track

def get_settings_from_testname(test):
    datalen_bytes = re.findall("\d*B", test)[0].replace("B", "")
    pub_count = re.findall("\d*P", test)[0].replace("P", "")
    sub_count = re.findall("\d*S", test)[0].replace("S", "")
    best_effort = len(re.findall("_be_", test)) > 0
    multicast = len(re.findall("_mc_", test)) > 0
    durability = re.findall("\ddur", test)[0].replace("dur", "")

    return datalen_bytes, pub_count, sub_count, best_effort, multicast, durability

datadir = "C:/Users/acwh025/Documents/Software Dev/PTST-Visualiser/summaries"

test_summaries = [os.path.join(datadir, _) for _ in os.listdir(datadir)]

df = pd.DataFrame(columns=[
    'datalen_bytes', 
    'pub_count',
    'sub_count',
    'best_effort',
    'multicast',
    'durability',
    'mean_latency_us',
    'mean_total_throughput_mbps',
    'mean_sample_rate',
    'samples_received',
    'samples_lost'
])

for i in track(range( len(test_summaries) ), description="Processing summaries..."):
    summary = test_summaries[i]
    i = test_summaries.index(summary)
    summary_df = pd.read_csv(summary)
    testname = os.path.basename(summary.replace("_summary.csv", ""))
    datalen_bytes, pub_count, sub_count, best_effort, multicast, durability = get_settings_from_testname(testname)

    data = [
        datalen_bytes,
        pub_count,
        sub_count,
        best_effort,
        multicast,
        durability,
        summary_df["latency_us"].mean(),
        summary_df['total_throughput_mbps'].mean(),
        summary_df['total_sample_rate'].mean(),
        summary_df['total_samples_received'].max(),
        summary_df['total_samples_lost'].max()
    ]

    df.loc[i] = data
    
# ? Cut the df so all columns have the same length
min_length = len(df[df.columns[0]])
for col in df.columns:
    col_length = len(df[col])
    if col_length < min_length:
        min_length = col_length

# ? Cut the dataframe to the shortest column
df = df.iloc[:min_length]

df.to_csv('df.csv', index=False)

Output()